# AutoAD - usage examples

AutoAD turns a mixed-type DataFrame into a numeric feature matrix, adds robust
univariate outlier counts and fits any [PyOD](https://github.com/yzhao062/pyod)
detector on it. This notebook walks through the typical tasks:

1. Example data
2. Quick start: fit and report
3. Scores and labels
4. What AutoAD decided
5. Telling AutoAD about your columns
6. Scoring new data
7. Feature importances
8. Choosing a detector
9. Tuning the preprocessing
10. Only the feature matrix, and scikit-learn integration
11. The building blocks on their own
12. Saving and loading a fitted model

**Setup:** run `pip install -e ".[examples]"` in the repository root. Without
installing, the next cell makes the source in `../src` importable.

In [1]:
import importlib.util
import sys
from pathlib import Path

if importlib.util.find_spec("autoad") is None:  # not installed: use the source checkout
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
import numpy as np
import pandas as pd
from pyod.models.iforest import IForest
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.pca import PCA

from autoad import AutoAD

## 1. Example data

500 synthetic employees with numeric, categorical, boolean and date columns, some
missing ages and four anomalies that we plant on purpose. All values are random, so
the data may contain a few natural outliers as well.

In [3]:
rng = np.random.default_rng(42)
n = 500
X = pd.DataFrame(
    {
        "employee_id": np.arange(1, n + 1),
        "age": rng.normal(40, 9, n).round(),
        "salary": rng.normal(62_000, 9_000, n).round(2),
        "years_at_company": rng.integers(0, 25, n),
        "department": rng.choice(["HR", "Engineering", "Finance", "Sales", "Marketing"], n),
        "city": rng.choice([f"City {i}" for i in range(60)], n),
        "is_manager": rng.random(n) < 0.15,
        "hire_date": (
            pd.Timestamp("2010-01-01") + pd.to_timedelta(rng.integers(0, 5000, n), unit="D")
        ).strftime("%Y-%m-%d"),
    }
)
X.loc[rng.choice(n, 15, replace=False), "age"] = np.nan

INJECTED = {
    7: "salary of 1.2 million",
    42: "age of 130",
    99: "40 years at the company at age 21",
    256: "negative salary",
}
X.loc[7, "salary"] = 1_200_000
X.loc[42, "age"] = 130
X.loc[99, ["age", "years_at_company"]] = [21, 40]
X.loc[256, "salary"] = -5_000
X.head()

,employee_id,age,salary,years_at_company,department,city,is_manager,hire_date
0,1,NaN,74274.76,4,Marketing,City 25,False,2019-06-21
1,2,31.0,70056.66,16,Marketing,City 28,False,2011-04-13
2,3,47.0,55524.68,10,Sales,City 14,False,2016-10-10
3,4,48.0,48477.47,8,Engineering,City 46,False,2022-02-27
4,5,22.0,35319.24,17,Sales,City 34,True,2019-09-27


## 2. Quick start: fit and report

Pass a detector, call `fit`, then `report`. The report is a copy of the input with
the anomaly information added; rows keep their original order.

In [4]:
ad = AutoAD(IForest(random_state=0)).fit(X)
report = ad.report(X)
report.sort_values("AD_score", ascending=False).head(8)

,employee_id,age,salary,years_at_company,department,city,is_manager,hire_date,AD_score,Tukey_Total,Tukey_Columns,MAD_Total,MAD_Columns
7,8,37.0,1200000.00,11,Finance,City 48,False,2012-09-25,0.164418,1,salary,1,salary
139,140,66.0,66720.57,3,Marketing,City 14,False,2015-11-02,0.132588,1,age,1,age
449,450,43.0,29164.28,24,Sales,City 55,False,2018-10-17,0.115827,1,salary,1,salary
256,257,44.0,-5000.00,8,Engineering,City 9,False,2012-09-11,0.112436,1,salary,1,salary
42,43,130.0,64228.99,22,HR,City 3,False,2020-10-07,0.111522,1,age,1,age
398,399,NaN,66309.88,0,Sales,City 31,True,2010-07-05,0.101594,0,,0,
4,5,22.0,35319.24,17,Sales,City 34,True,2019-09-27,0.100903,1,salary,0,
248,249,66.0,58654.55,21,HR,City 15,False,2017-04-24,0.098827,1,age,1,age


| Column | Meaning |
| --- | --- |
| `AD_score` | score of the detector, higher means more anomalous |
| `Tukey_Total` | number of the row's numeric values outside Tukey's fences |
| `MAD_Total` | number of the row's numeric values with a modified z-score above 3.5 |
| `Tukey_Columns`, `MAD_Columns` | the names of these columns |

The `*_Columns` explain *why* a row stands out. Here are the planted anomalies:

In [5]:
columns = ["age", "salary", "years_at_company", "AD_score", "Tukey_Columns", "MAD_Columns"]
report.loc[list(INJECTED), columns].assign(planted=pd.Series(INJECTED))

,age,salary,years_at_company,AD_score,Tukey_Columns,MAD_Columns,planted
7,37.0,1200000.00,11,0.164418,salary,salary,salary of 1.2 million
42,130.0,64228.99,22,0.111522,age,age,age of 130
99,21.0,46025.34,40,0.061506,years_at_company,,40 years at the company at age 21
256,44.0,-5000.00,8,0.112436,salary,salary,negative salary


## 3. Scores and labels

`decision_function` returns the scores as an array. `predict` turns them into 0/1
labels: a row is labelled 1 if it scores higher than the top share of the training
data given by the detector's `contamination` (PyOD's default is 10 %).

In [6]:
scores = ad.decision_function(X)
labels = ad.predict(X)
print(f"contamination=0.1:  {labels.sum()} of {len(X)} rows labelled as anomalies")

strict = AutoAD(IForest(contamination=0.01, random_state=0)).fit(X)
print("contamination=0.01:", X.index[strict.predict(X) == 1].tolist())

contamination=0.1:  50 of 500 rows labelled as anomalies
contamination=0.01: [7, 42, 139, 256, 449]


## 4. What AutoAD decided

Every decision is stored on the fitted object. AutoAD also logs them via the
`autoad` logger (enable with `logging.basicConfig(level=logging.INFO)`).

- `employee_id` is dropped because it looks like an ID (unique and named `*_id`).
- `department` has few values and gets a binary code, `city` has 60 and is
  frequency-encoded.
- `hire_date` is stored as text but recognised as a date.

In [7]:
print("Types:  ", ad.column_types_)
print("Dropped:", ad.dropped_columns_)
print("Groups: ", ad.column_groups_)
print("Features the detector sees:", list(ad.get_feature_names_out()))

Types:   {'employee_id': 'numeric', 'age': 'numeric', 'salary': 'numeric', 'years_at_company': 'numeric', 'department': 'categorical', 'city': 'categorical', 'is_manager': 'bool', 'hire_date': 'datetime'}
Dropped: {'employee_id': 'id'}
Groups:  {'numeric': ['age', 'salary', 'years_at_company'], 'bool': ['is_manager'], 'binary': ['department'], 'frequency': ['city'], 'datetime': ['hire_date']}
Features the detector sees: ['age', 'salary', 'years_at_company', 'Tukey_Total', 'MAD_Total', 'is_manager', 'department_0', 'department_1', 'department_2', 'city_FREQ', 'hire_date_YEAR', 'hire_date_MONTH', 'hire_date_DAY', 'hire_date_WKDAY', 'missing_count']


## 5. Telling AutoAD about your columns

Type inference is conservative: a text column becomes a date only if almost all
values are dates, and a number only if all values are numbers. Real exports are
messier. Here dates and amounts contain placeholders, and a free-text comment
should not be used at all:

- `datetime_columns` forces date parsing (unparsable values become missing),
- `numerical_columns` forces numbers,
- `exclude_columns` ignores columns completely.

In [8]:
orders = pd.DataFrame(
    {
        "order_date": ["2024-01-03", "2024-02-17", "tbd", "2024-03-21", "2024-04-02", "unknown"],
        "amount": ["120.50", "99.90", "130.00", "unknown", "9999.00", "101.10"],
        "status": ["paid", "paid", "open", "paid", "paid", "open"],
        "comment": ["", "call back", "", "urgent", "", "fragile"],
    }
)
automatic = AutoAD().fit(orders)
explicit = AutoAD(
    datetime_columns=["order_date"],
    numerical_columns=["amount"],
    exclude_columns=["comment"],
).fit(orders)
pd.DataFrame({"automatic": automatic.column_types_, "explicit": explicit.column_types_})

,automatic,explicit
order_date,categorical,datetime
amount,categorical,numeric
status,categorical,categorical
comment,categorical,NaN


With `amount` treated as a number, both univariate methods find the 9,999.00
order. `tbd` and `unknown` are now missing values; the detector sees them through
the `missing_count` feature.

In [9]:
explicit.report(orders)

,order_date,amount,status,comment,Tukey_Total,Tukey_Columns,MAD_Total,MAD_Columns
0,2024-01-03,120.50,paid,,0,,0,
1,2024-02-17,99.90,paid,call back,0,,0,
2,tbd,130.00,open,,0,,0,
3,2024-03-21,unknown,paid,urgent,0,,0,
4,2024-04-02,9999.00,paid,,1,amount,1,amount
5,unknown,101.10,open,fragile,0,,0,


## 6. Scoring new data

New rows go through exactly the same steps as the training data. Categories that
never occurred during training and missing values in columns that had none are
handled and show up in the score.

In [10]:
X_new = pd.DataFrame(
    {
        "employee_id": [501, 502, 503],
        "age": [35, np.nan, 29],
        "salary": [61_000, 64_000, 950_000],
        "years_at_company": [4, 10, 2],
        "department": ["Engineering", "Legal", "Sales"],  # "Legal" is new
        "city": ["City 3", "City 99", "City 12"],  # "City 99" is new
        "is_manager": [False, True, False],
        "hire_date": ["2019-04-01", "2015-09-15", "2023-01-10"],
    }
)
ad.report(X_new)

,employee_id,age,salary,years_at_company,department,city,is_manager,hire_date,AD_score,Tukey_Total,Tukey_Columns,MAD_Total,MAD_Columns
0,501,35.0,61000,4,Engineering,City 3,False,2019-04-01,-0.064243,0,,0,
1,502,NaN,64000,10,Legal,City 99,True,2015-09-15,0.084866,0,,0,
2,503,29.0,950000,2,Sales,City 12,False,2023-01-10,0.151023,1,salary,1,salary


The columns must match the training data (their order does not matter). A
mismatch is reported before anything is computed:

In [11]:
try:
    ad.transform(X_new.drop(columns="city").assign(bonus=0))
except ValueError as error:
    print(error)

X does not have the columns seen during fit. Missing: ['city']. Unexpected: ['bonus'].


## 7. Feature importances

Detectors with a `feature_importances_` attribute, such as Isolation Forest, expose
them per feature; other detectors raise an `AttributeError`. For Isolation Forest
they show how often the trees split on a feature, which says more about the model
than about a single row. To see why a particular row stands out, look at its
`Tukey_Columns` and `MAD_Columns` in the report.

In [12]:
ad.feature_importances.head(8)

,Feature,Importance
0,city_FREQ,0.110517
1,hire_date_DAY,0.109360
2,years_at_company,0.103568
3,salary,0.095899
4,hire_date_WKDAY,0.094069
5,age,0.093351
6,hire_date_MONTH,0.089124
7,hire_date_YEAR,0.085634


## 8. Choosing a detector

Any PyOD detector can be plugged in. A quick way to compare them on your data is to
look at where known anomalies end up. Rank 1 is the most anomalous of 500 rows.

In [13]:
detectors = {
    "IForest": IForest(random_state=0),
    "LOF": LOF(),
    "KNN": KNN(),
    "PCA": PCA(random_state=0),
}
ranks = {}
for name, detector in detectors.items():
    detector_scores = AutoAD(detector).fit(X).decision_function(X)
    rank = pd.Series(detector_scores, index=X.index).rank(ascending=False).astype(int)
    ranks[name] = rank[list(INJECTED)]
pd.DataFrame(ranks).assign(planted=pd.Series(INJECTED))

,IForest,LOF,KNN,PCA,planted
7,1,1,1,1,salary of 1.2 million
42,5,2,2,2,age of 130
99,14,4,6,7,40 years at the company at age 21
256,4,3,3,6,negative salary


## 9. Tuning the preprocessing

The defaults work without configuration; these parameters change the feature matrix:

| Parameter | Default | Effect |
| --- | --- | --- |
| `tukey_factor`, `mad_threshold` | 1.5, 3.5 | how far out a value must be to count in `Tukey_Total`/`MAD_Total` |
| `max_binary_categories` | 20 | above this, categories are frequency-encoded |
| `missing_indicator` | `"count"` | `"columns"`: one indicator per column, `None`: none |
| `scaler` | `"robust"` | `"standard"`, `"minmax"`, `None` or any scikit-learn transformer |
| `drop_constant`, `drop_id_columns` | `True` | keep constant or ID-like columns with `False` |

In [14]:
custom = AutoAD(
    IForest(random_state=0),
    tukey_factor=3.0,
    mad_threshold=5.0,
    max_binary_categories=3,
    missing_indicator="columns",
    scaler="standard",
).fit(X)
print(
    "Rows with a Tukey finding, default vs. factor 3:",
    (report["Tukey_Total"] > 0).sum(),
    "vs.",
    (custom.report(X)["Tukey_Total"] > 0).sum(),
)
print("Features:", list(custom.get_feature_names_out()))

Rows with a Tukey finding, default vs. factor 3: 13 vs. 3
Features: ['age', 'salary', 'years_at_company', 'Tukey_Total', 'MAD_Total', 'is_manager', 'department_FREQ', 'city_FREQ', 'hire_date_YEAR', 'hire_date_MONTH', 'hire_date_DAY', 'hire_date_WKDAY', 'age_MISSING', 'salary_MISSING', 'years_at_company_MISSING', 'is_manager_MISSING', 'department_MISSING', 'city_MISSING', 'hire_date_MISSING']


## 10. Only the feature matrix, and scikit-learn integration

Without a detector, AutoAD is a preprocessing step: `fit_transform` returns the
scaled numeric matrix, ready for any other model. AutoAD is a regular scikit-learn
estimator, so `get_params`, `set_params` and `clone` work as usual.

In [15]:
features = AutoAD().fit_transform(X)
print(features.shape)
features.head()

(500, 15)


,age,salary,years_at_company,Tukey_Total,MAD_Total,is_manager,department_0,department_1,department_2,city_FREQ,hire_date_YEAR,hire_date_MONTH,hire_date_DAY,hire_date_WKDAY,missing_count
0,0.000000,1.010685,-0.583333,0.0,0.0,0.0,0.0,0.0,0.0,-0.2,0.285714,0.0,0.357143,0.25,1.0
1,-0.818182,0.661136,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.6,-0.857143,-0.4,-0.214286,-0.25,0.0
2,0.636364,-0.543115,-0.083333,0.0,0.0,0.0,0.0,1.0,-1.0,1.4,-0.142857,0.8,-0.428571,-0.75,0.0
3,0.727273,-1.127110,-0.250000,0.0,0.0,0.0,0.0,1.0,0.0,1.2,0.714286,-0.8,0.785714,0.75,0.0
4,-1.636364,-2.217520,0.500000,1.0,0.0,1.0,0.0,1.0,-1.0,0.0,0.285714,0.6,0.785714,0.25,0.0


In [16]:
from sklearn.base import clone

variant = clone(ad).set_params(mad_threshold=4.0, detector=KNN(n_neighbors=10))
variant.fit(X)
variant.get_params()["mad_threshold"], type(variant.detector_).__name__

(4.0, 'KNN')

## 11. The building blocks on their own

The univariate methods are scikit-learn transformers as well and can be used
directly, for example to look at the fences and scores of a single column.

In [17]:
from autoad.statistical import MedianAbsoluteDeviation, TukeyTransformer

salary = X[["salary"]]
tukey = TukeyTransformer().fit(salary)
mad = MedianAbsoluteDeviation(scores_output=True).fit(salary)
print(f"Tukey fences for salary: {tukey.lower_[0]:,.0f} to {tukey.upper_[0]:,.0f}")

rows = [0, 1, 7, 256]
pd.DataFrame(
    {
        "salary": X.loc[rows, "salary"],
        "outside_fences": tukey.transform(salary.loc[rows])[:, 0],
        "modified_z_score": mad.transform(salary.loc[rows])[:, 0].round(1),
    }
)

Tukey fences for salary: 37,373 to 85,642


,salary,outside_fences,modified_z_score
0,74274.76,0,1.4
1,70056.66,0,0.9
7,1200000.00,1,131.3
256,-5000.00,1,-7.7


## 12. Saving and loading a fitted model

A fitted AutoAD object, including its detector, can be stored with `joblib` and
used later to score new data.

In [18]:
import tempfile

import joblib

path = Path(tempfile.gettempdir()) / "autoad_model.joblib"
joblib.dump(ad, path)
restored = joblib.load(path)
np.allclose(restored.decision_function(X_new), ad.decision_function(X_new))

True